In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

print("PageIndex key :", "Success" if PAGEINDEX_API_KEY else "Failed")
print("Groq key :", "Success" if PAGEINDEX_API_KEY else "Failed")

PageIndex key : Success
Groq key : Success


In [2]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
llm = Groq()

print("Pageindex client ready")
print("Groq client ready")

d:\agenticai\.venv\Lib\site-packages\pageindex\__init__.py:9: UserWarning: You're on Python 3.11, so an older pageindex (0.2.8.1) was installed. pageindex 0.2.10+ requires Python 3.10+.
  warnings.warn(_OLDER_INSTALL + " pageindex 0.2.10+ requires Python 3.10+.")


Pageindex client ready
Groq client ready


In [3]:
PDF_PATH = "./sampleDocument.pdf"

print("Uploading!!")
res = pi_client.submit_document(PDF_PATH)
doc_id = res["doc_id"]

print("Upload Success!!")
print(f"Document ID: {doc_id}")

Uploading!!
Upload Success!!
Document ID: pi-cmw07hu61003q0cnnsptfbnh2


In [4]:
import time

print("Building tree index")

while True:
    state_result = pi_client.get_document(doc_id)
    status = state_result.get("status")
    print(f"Status: {status}")

    if status == "completed":
        print("Tree Index ready")
        break
    elif status == "failed":
        print("Failed")
        break

    time.sleep(5)


Building tree index
Status: processing
Status: processing
Status: processing
Status: processing
Status: processing
Status: completed
Tree Index ready


In [5]:
import json

tree_res = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_res.get("result", [])

print(f"Top level sections: {len(pageindex_tree)}")
print("Raw Tree [first node]")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top level sections: 14
Raw Tree [first node]
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "start_index": 1,
  "end_index": 3,
  "summary": "This document is the title, quote, and table of contents for Andrew Ng's guide on building an AI career, outlining chapters on technical learning, project building, job searching, and professional growth.",
  "text": "A Simple Guide\n\nDeepLearning.AI\n\nHow to Build Your Career in AI\n\nCollected Insights from Andrew Ng\n\nFounder, DeepLearning.AI\n\n![img-0.jpeg](img-0.jpeg)\n\n\"AI is the new electricity. It will transform and improve all areas of human life.\"\n\nAndrew Ng\n\nTable of Contents\n\nIntroduction: Coding AI is the New Literacy.\n\nChapter 1: Three Steps to Career Growth.\n\n![img-1.jpeg](img-1.jpeg)\n\nLEARNING\n\nChapter 2: Learning Technical Skills for a Promising AI Career.\n\nChapter 3: Should You Learn Math to Get a Job in AI?\n\n![img-2.jpeg](img-2.jpeg)\n\nPROJECTS\n\nChapter 4: Scoping Successful AI Pro

In [6]:
def print_tree(nodes, indent=0):
    """Recursively prints tree titles for visual overview."""
    for node in nodes:
        prefix = "  "*indent + ("~~" if indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node['nodes'], indent+1)

print("Full Document Stuructre: \n")
print_tree(pageindex_tree)

Full Document Stuructre: 

[0000] Preface (p.1)
[0001] Introduction--- (p.4)
[0002] Coding AI Is the New Literacy (p.4)
[0003] Three Steps to Career Growth (p.5)
  ~~[0004] Three Steps to Career Growth (intro) (p.5)
  ~~[0005] Learning foundational skills is a career-long process: (p.7)
  ~~[0006] Working on projects often means collaborating with stakeholders who lack expertise in AI: (p.7)
  ~~[0007] Inconsistent opinions on AI skills and jobs roles: (p.7)
[0008] Learning Technical Skills for a Promising AI Career (p.8)
  ~~[0009] Learning Technical Skills for a Promising AI Career (intro) (p.8)
  ~~[0010] This is a lot to learn! (p.10)
  ~~[0011] The Best Way to Build a New Habit (p.11)
[0012] Should You Learn Math to Get a Job in AI? (p.12)
[0013] Scoping Successful AI Projects (p.14)
  ~~[0014] Scoping Successful AI Projects (intro) (p.14)
  ~~[0015] Step 1 (p.15)
  ~~[0016] Step 2 (p.15)
  ~~[0017] Step 3 (p.16)
  ~~[0018] Step 4 (p.16)
  ~~[0019] Step 5 (p.16)
[0020] Finding Pro

In [7]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total+= count_nodes(n['nodes'])

    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")

Total nodes in tree: 46


In [8]:
def llm_tree_search(query: str, tree: list, model="openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex Retrieval:
    Sends the query + document tree to an LLM. 
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and node_list (node IDS)
    """

    def compress(tree):
        out = []
        for n in tree:
            entry = {
                "node_id" : n['node_id'],
                "title" : n['title'],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)

        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like Table of contents).
    Your task: identify which node IDs are most likely to contain the answer to the query.
    Think step-by-step about which sections are relevant.
    
    Query:{query}

    Document Tree:
    {json.dumps(compressed_tree,indent = 2)}

    Reply ONLY in this exact JSON format:
    {{
    "thinking" : "<your step-by-step reasoning>",
    "node_list": ["node_id1", "nodeid2"]
    }}
    
    """

    response = llm.chat.completions.create(
        model = model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)


In [9]:
query = "How to show some skill progression?"

print(f"Query: {query}")
result = llm_tree_search(query,pageindex_tree)

print("LLM reasoning:")
print(result.get("thinking","N/A"))
print()
print("Selected Node IDs: ", result.get("node_list",[]))

Query: How to show some skill progression?
LLM reasoning:
The query asks how to demonstrate skill progression. The section titled 'Building a Portfolio of Projects that Shows Skill Progression' directly addresses this, and its subsections (intro, class projects, personal projects, creating value, rising scope and complexity) elaborate on ways to showcase growing abilities. These nodes are the most relevant, while other sections on learning foundational skills are broader and less specific to showing progression.

Selected Node IDs:  ['0024', '0025', '0026', '0027', '0028', '0029']


In [10]:
def find_node_by_id(tree:list , target_ids: list) -> list:
    """ Recursively walk the tree and collect nodes matching the target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_node_by_id(node['nodes'], target_ids))

    return found

In [20]:
def generate_ans(query:str, nodes:list , model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generated a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """

    if not nodes:
        return "No relevant node found in the document"

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}] \n{node.get('text', 'Content not available')}"
        )

    context = "\n\n\n----\n\n".join(context_parts)

    prompt = f"""
    You are an expert document analyst.
    Answer the following question using ONLY the provided context.
    For every claim you make, cite the section title and page number in parantheses.
    Be concise and precise.

    Question: {query}

    Context: {context}

    Answer:
"""
    response = llm.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content":prompt}]
    )

    return response.choices[0].message.content

In [21]:
def vectorlessRag(query: str, tree: list, verbose: bool=True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search -> Finds relevant node_ids
    Setp 2: Node Retreival -> fetches section content
    Step 3: Answer Generation -> produces cited answer
    """

    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")

    search_result = llm_tree_search(query,tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking','')[:200]}...")
        print(f"Retreived node IDs: {node_ids}")

    nodes = find_node_by_id(tree, node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")

    answer = generate_ans(query,nodes)

    if verbose: 
        print(f"Answer: \n{answer}")

    return answer

In [22]:
answer = vectorlessRag(
    query="What are the key things to build future in AI?",
    tree = pageindex_tree
)

Query: What are the key things to build future in AI?

 Reasoning: The query asks for the key things to build the future in AI. Sections that explicitly list 'keys' or steps are most relevant. The chapter titled 'Keys to Building a Career in AI' (node 0035) and its s...
Retreived node IDs: ['0035', '0036', '0037', '0038', '0039', '0040', '0041']
Sections found: ['Keys to Building a Career in AI', 'Keys to Building a Career in AI (intro)', '1. Teamwork:', '2. Networking:', '3. Job search', '4. Personal discipline', '5. Altruism']
Answer: 
The future‑focused AI career rests on five core practices:

1. **Teamwork** – collaborating effectively, influencing and being influenced by others, and strong interpersonal/communication skills are essential for large‑scale projects. (Section “1. Teamwork:”, p. 35)  
2. **Networking** – cultivating a professional community (rather than just a personal contact list) helps you meet allies and obtain advice when needed. (Section “2. Networking:”, p. 35) 

In [23]:
test_queries = [
    "What are the key skills to learn in AI?",
    "How to find projects to supplement career goals?",
    "How to overcome imposter syndrome?"
]

for q in test_queries:
    print()
    ans = vectorlessRag(q,pageindex_tree,verbose=False)
    print(f"Question: {q}")
    print(f"Answer: {ans[:300]}...")
    print("="*55)


Question: What are the key skills to learn in AI?
Answer: The core AI skill set consists of :

* **Foundations of machine learning and deep learning** – the basic algorithms, model‑training pipelines, and the evolving tools that underpin AI work (Learning foundational skills is a career‑long process:, p. 7).  
* **Mathematical fundamentals** – linear algeb...

Question: How to find projects to supplement career goals?
Answer: **How to find projects that supplement your career goals**

1. **Start small and build momentum** – Begin with modest, responsible projects that let you learn and gradually take on larger work as your skills grow. Early successes make it easier to generate better ideas and persuade collaborators. *(...

Question: How to overcome imposter syndrome?
Answer: **Ways to overcome imposter syndrome (as outlined in the text):**  

1. **Find supportive mentors or peers.**  
   - Attend events (e.g., Pie & AI), use discussion boards, and actively seek mentors who encourage